# Email Finder — YouTube 19k list (NO Perplexity)

Runs the **email_finder agent** from `ai_agents_service` over the YouTube-channel
sheet, but with the **expensive Perplexity research node removed** to cut cost.
The Facebook/Apify crawler is removed too (cheapest setting).

**Pipeline kept (everything but Perplexity + fb_crawler):**
`canonical_builder → validate_existing_email → youtube_about_enricher →
website_guesser (GPT website finder) → discover_urls → crawl_page (crawler) →
resolve_best_email`

We do this by building a notebook-local graph that reuses **every** routing
function from the agent unchanged, and simply remaps each `perplexity_discovery`
and `fb_crawler` edge to `END`. Nothing in the shared agent repo is modified.

> Perplexity is never *called*, but `PERPLEXITY_API_KEY` must still be present in
> `.env` because importing the agent's `graph` module imports the (unused) node.

**Source sheet shape** (`Sheet1`, ~19,428 rows): `title, channel_handle, email,
subscriber_count, video_count, niche, category, country, website_url,
twitter_url, instagram_url, tiktok_url, facebook_url, linkedin_url`.
Notable: `email` is mostly empty or the literal `"x"`; `website_url` is ~84%
YouTube infra junk (thumbnails / playback URLs) — both are sanitized below.

**Run order:** run cell-by-cell. The runner defaults to `LIMIT=10` + `DRY_RUN=True`
so nothing is written until you've eyeballed the output.

In [ ]:
from __future__ import annotations

import json
import os
import asyncio
from collections import Counter
from datetime import datetime
from urllib.parse import urlparse

import sys
sys.path.insert(0, "/Users/utkarshumang/my_projects/ai-agents-service")
sys.path.insert(0, "../..")

# Load env BEFORE importing ai_agents (its bootstrap validates required keys at
# import time — including PERPLEXITY_API_KEY, even though we never call it).
from dotenv import load_dotenv
load_dotenv("/Users/utkarshumang/my_projects/lead-enricher-ai-be/.env")

import nest_asyncio

from langgraph.graph import END, StateGraph

from ai_agents.agents.email_finder import SourceType, LeadStatus
from ai_agents.agents.email_finder.graph_state import EmailFinderGraphState
# Reuse the agent's own node callables + routing functions verbatim.
from ai_agents.agents.email_finder.graph import (
    run_canonical_builder,
    route_after_canonical,
    route_after_validate,
    route_after_discover,
    route_after_resolve,
    route_after_youtube_enrich,
    route_after_website_guess,
    validate_existing_email_node_async,
    discover_urls_node_async,
    crawl_page_node_async,
    resolve_best_email_node_async,
    youtube_about_enricher_node_async,
    website_guesser_node_async,
    run_single_async,
)
from google_utils.google_sheet import GoogleSheetService

In [ ]:
# ── Config ─────────────────────────────────────────────────────────────────
SPREADSHEET_URL = "https://docs.google.com/spreadsheets/d/1KywXf2BkClVkW6TMXKp_IQ40l6FGpG_Q9aQKiCRTf2A/edit"
SOURCE_SHEET = "Sheet1"
OUTPUT_SHEET = "Email_Finder"

SOURCE_TYPE  = SourceType.OTHER     # generic YouTube channels
YOUTUBE_LIST = True                 # enable the free youtube_about_enricher node

CHECKPOINT_FILE = "email_finder_youtube_19k_checkpoint.json"
BATCH_SIZE  = 50
CONCURRENCY = 12            # leads in flight at once (bounded by per-lead timeout)

# Speed guards — the crawler is the slow part. Without these a single slow site
# can burn ~6 min/page (180s x 2 attempts) and stall a whole wave.
PER_LEAD_TIMEOUT_S   = 60   # hard cap per channel; on timeout -> not_found, move on
CRAWL_HARD_TIMEOUT_S = 25   # override the agent's 180s crawl_page hard cap

# Skip rows that already have a real (@) email.
SKIP_ROWS_WITH_EMAIL = True

## The cost change — a Perplexity-free (and fb_crawler-free) graph

`build_graph_no_perplexity()` is a copy of the agent's `build_graph()` with two
nodes dropped. The routing functions are imported and reused **unchanged** — we
only change the edge *maps* so any router that returns `"perplexity_discovery"`
or `"fb_crawler"` lands on `END` instead.

In [ ]:
def build_graph_no_perplexity():
    """Agent graph minus perplexity_discovery and fb_crawler (both → END)."""
    g = StateGraph(EmailFinderGraphState)

    g.add_node("canonical_builder", run_canonical_builder)
    g.add_node("validate_existing_email", validate_existing_email_node_async)
    g.add_node("discover_urls", discover_urls_node_async)
    g.add_node("crawl_page", crawl_page_node_async)
    g.add_node("resolve_best_email", resolve_best_email_node_async)
    g.add_node("youtube_about_enricher", youtube_about_enricher_node_async)
    g.add_node("website_guesser", website_guesser_node_async)

    g.set_entry_point("canonical_builder")

    g.add_conditional_edges("canonical_builder", route_after_canonical, {
        "validate_existing_email": "validate_existing_email",
        "discover_urls": "discover_urls",
        "youtube_about_enricher": "youtube_about_enricher",
        "website_guesser": "website_guesser",
        "perplexity_discovery": END,          # removed → stop
    })
    g.add_conditional_edges("validate_existing_email", route_after_validate, {
        END: END,
        "discover_urls": "discover_urls",
        "youtube_about_enricher": "youtube_about_enricher",
        "website_guesser": "website_guesser",
        "perplexity_discovery": END,          # removed → stop
    })
    g.add_conditional_edges("discover_urls", route_after_discover, {
        "crawl_page": "crawl_page",
        "resolve_best_email": "resolve_best_email",
    })
    g.add_edge("crawl_page", "resolve_best_email")
    g.add_conditional_edges("resolve_best_email", route_after_resolve, {
        END: END,
        "fb_crawler": END,                    # removed → stop
        "perplexity_discovery": END,          # removed → stop
    })
    g.add_conditional_edges("youtube_about_enricher", route_after_youtube_enrich, {
        END: END,
        "discover_urls": "discover_urls",
        "website_guesser": "website_guesser",
        "perplexity_discovery": END,          # removed → stop
    })
    g.add_conditional_edges("website_guesser", route_after_website_guess, {
        "discover_urls": "discover_urls",
        "perplexity_discovery": END,          # removed → stop
    })
    return g.compile()


# Build once and reuse across the session.
NO_PPLX_GRAPH = build_graph_no_perplexity()
print("Compiled Perplexity-free graph. Nodes:",
      sorted(NO_PPLX_GRAPH.get_graph().nodes.keys()))

In [ ]:
import time
from concurrent.futures import ThreadPoolExecutor, as_completed
import ai_agents.agents.email_finder.nodes.crawl_page as _crawl_mod

# Tighten the crawler's hard wall-clock cap (agent default = 180s x 2 attempts).
_crawl_mod._CRAWL_HARD_TIMEOUT_S = CRAWL_HARD_TIMEOUT_S
print(f"crawl hard timeout set to {_crawl_mod._CRAWL_HARD_TIMEOUT_S}s")

# WHY THREADS, NOT asyncio.gather:
# Several agent nodes do BLOCKING sync I/O inside async functions — most notably
# youtube_about_enricher (httpx.Client GET to YouTube, 30s) and url_discovery's
# sitemap fetches. In ONE event loop those block everything, so asyncio.gather
# collapses to ~1 lead at a time and the per-lead wait_for can't even fire.
# Running each lead in its OWN thread (own event loop) is what truly parallelizes
# them. This mirrors the repo's benchmark-discovery-models notebook.

LEAD_TIMINGS = []


def _lead_label(row, i):
    return row.get("Channel Name") or f"row-{i}"


def _run_lead(row, i, total, source=None):
    """Run ONE lead in this thread, with its own event loop + soft timeout."""
    t0 = time.monotonic()

    async def _go():
        return await asyncio.wait_for(
            NO_PPLX_GRAPH.ainvoke({
                "raw_row": row,
                "source_type": SOURCE_TYPE.value,
                "youtube_list": YOUTUBE_LIST,
                "source": source,
            }),
            timeout=PER_LEAD_TIMEOUT_S,
        )

    try:
        res = asyncio.run(_go())
    except asyncio.TimeoutError:
        res = {"status": LeadStatus.EMAIL_NOT_FOUND.value,
               "errors": [f"per-lead timeout {PER_LEAD_TIMEOUT_S}s"],
               "raw_row": row, "nodes_executed": ["__timeout__"]}
    except Exception as e:
        res = {"status": LeadStatus.FAILED.value, "errors": [str(e)[:200]],
               "raw_row": row, "nodes_executed": []}
    res["_elapsed_s"] = round(time.monotonic() - t0, 1)
    return res


def run_batch_no_pplx(rows, source_type=SOURCE_TYPE, concurrency=CONCURRENCY,
                      youtube_list=YOUTUBE_LIST, source=None):
    """Run leads in a thread pool (true parallelism for the blocking nodes)."""
    total = len(rows)
    results = [None] * total
    done = 0
    with ThreadPoolExecutor(max_workers=concurrency) as ex:
        futs = {ex.submit(_run_lead, r, i + 1, total, source): i
                for i, r in enumerate(rows)}
        for fut in as_completed(futs):
            i = futs[fut]
            try:
                res = fut.result()
            except Exception as e:
                res = {"status": LeadStatus.FAILED.value, "errors": [str(e)[:200]],
                       "raw_row": rows[i], "nodes_executed": []}
            results[i] = res
            LEAD_TIMINGS.append(res.get("_elapsed_s", 0))
            done += 1
            email = (res.get("best_email") or {}).get("email", "")
            path = " -> ".join(res.get("nodes_executed") or [])
            print(f"  [{done}/{total}] {res.get('status','?'):16} "
                  f"{res.get('_elapsed_s', '?'):>5}s  {(email or '(none)'):28} "
                  f"{_lead_label(rows[i], i+1)[:22]:22} | {path}")
    return results


def run_single_no_pplx(raw_row, source_type=SOURCE_TYPE, youtube_list=YOUTUBE_LIST,
                       source=None):
    return run_batch_no_pplx([raw_row], source_type, concurrency=1,
                             youtube_list=youtube_list, source=source)[0]

## Data hygiene for this sheet

Three sheet-specific cleanups before the agent sees a row:
- **`email`** — keep only if it looks like a real address (drops `"x"` / blanks).
- **`website_url`** — drop YouTube/Google infra hosts (thumbnails, playback,
  avatar CDNs) so the crawler never wastes a fetch on a `.jpg`.
- **`channel_handle`** — turn a `UC…` id or `@handle` into a real YouTube URL so
  `youtube_about_enricher` can read the About page.

In [ ]:
_INFRA_HOSTS = (
    "ytimg.com", "googlevideo.com", "youtube.com", "youtu.be", "google.com",
    "gstatic.com", "ggpht.com", "googleusercontent.com",
)


def _host(u: str) -> str:
    u = (u or "").strip()
    if not u:
        return ""
    if "://" not in u:
        u = "http://" + u
    try:
        return (urlparse(u).hostname or "").lower()
    except Exception:
        return ""


def _is_infra_host(h: str) -> bool:
    return any(h == d or h.endswith("." + d) for d in _INFRA_HOSTS)


def sanitize_website(u: str) -> str:
    """Return a real external website, or '' for YouTube/Google infra junk."""
    h = _host(u)
    if not h or _is_infra_host(h):
        return ""
    return (u or "").strip()


def is_real_email(e: str) -> bool:
    e = (e or "").strip().lower()
    return "@" in e and "." in e.split("@")[-1] and e not in {"x", "n/a", "na"}


def build_channel_url(handle: str) -> str:
    h = (handle or "").strip()
    if not h:
        return ""
    if h.startswith("http"):
        return h
    if h.startswith("UC"):
        return f"https://www.youtube.com/channel/{h}"
    if h.startswith("@"):
        return f"https://www.youtube.com/{h}"
    return f"https://www.youtube.com/@{h}"

In [ ]:
def build_raw_row(row: dict) -> dict:
    """Map a Sheet1 row → raw_row dict for the email finder.

    The canonical_builder LLM maps these keys → CanonicalLead:
      Channel Name / Brand Name -> identity
      YouTube / Channel URL     -> social_links.youtube (also gates the enricher)
      Email                     -> existing_email (only added when real)
      Website                   -> website (only added when not infra junk)
      Twitter/Instagram/...     -> social_links
    Underscore-prefixed keys are bookkeeping and ignored by the LLM.
    """
    title  = (row.get("title") or "").strip()
    handle = (row.get("channel_handle") or "").strip()
    channel_url   = build_channel_url(handle)
    existing_email = (row.get("email") or "").strip()
    existing_email = existing_email if is_real_email(existing_email) else ""
    website = sanitize_website(row.get("website_url"))

    raw = {
        "Channel Name": title,
        "Brand Name":   title,
        "YouTube":      channel_url,
        "Channel URL":  channel_url,
        "Country":      (row.get("country") or "").strip(),
        "Topic":        (row.get("niche") or row.get("category") or "").strip(),
        "Twitter":      (row.get("twitter_url") or "").strip(),
        "Instagram":    (row.get("instagram_url") or "").strip(),
        "Facebook":     (row.get("facebook_url") or "").strip(),
        "LinkedIn":     (row.get("linkedin_url") or "").strip(),
        "_handle":       handle,
        "_channel_url":  channel_url,
        "_existing_email": existing_email,
    }
    # Only surface these to the LLM when they are genuinely useful.
    if existing_email:
        raw["Email"] = existing_email
    if website:
        raw["Website"] = website
    return raw

In [ ]:
def collect_rows(sheet_service, spreadsheet_id, already_processed):
    """Read SOURCE_SHEET, keep rows needing an email, dedupe + skip processed."""
    ok, df = sheet_service.get_sheet_data(spreadsheet_id, SOURCE_SHEET)
    assert ok, f"Failed to read sheet: {df}"

    stats = Counter()
    seen = set(already_processed)
    rows = []
    for _, r in df.iterrows():
        stats["total"] += 1
        rd = r.to_dict()
        if SKIP_ROWS_WITH_EMAIL and is_real_email(rd.get("email")):
            stats["has_email_skipped"] += 1
            continue
        raw = build_raw_row(rd)
        key = raw["_handle"] or raw["_channel_url"] or raw["Channel Name"]
        if not key:
            stats["no_key"] += 1
            continue
        if key in seen:
            stats["dup_or_processed"] += 1
            continue
        seen.add(key)
        raw["_dedup_key"] = key
        rows.append(raw)

    print(f"Collection summary for '{SOURCE_SHEET}':")
    print(f"  Total rows scanned:           {stats['total']}")
    print(f"  Skipped (already had email):  {stats['has_email_skipped']}")
    print(f"  Skipped (no channel key):     {stats['no_key']}")
    print(f"  Skipped (dup / processed):    {stats['dup_or_processed']}")
    print(f"  Channels to process:          {len(rows)}")
    return rows

In [ ]:
# ── Checkpoint (resumable, keyed by channel handle) ──────────────────────────
def load_checkpoint(spreadsheet_id):
    if os.path.exists(CHECKPOINT_FILE):
        with open(CHECKPOINT_FILE) as f:
            data = json.load(f)
        if data.get("spreadsheet_id") == spreadsheet_id:
            print(f"Resuming — {len(data['processed_keys'])} channels already done")
            return data
    return {
        "spreadsheet_id": spreadsheet_id,
        "processed_keys": [],
        "found_count": 0, "not_found_count": 0, "failed_count": 0,
        "last_updated": None,
    }


def save_checkpoint(cp):
    cp["last_updated"] = datetime.now().isoformat()
    with open(CHECKPOINT_FILE, "w") as f:
        json.dump(cp, f, indent=2)


def clear_checkpoint():
    if os.path.exists(CHECKPOINT_FILE):
        os.remove(CHECKPOINT_FILE)
        print("Checkpoint cleared.")

In [ ]:
# ── Output sheet helpers ─────────────────────────────────────────────────────
OUTPUT_HEADERS = [
    "channel_name", "channel_url", "existing_email", "found_email",
    "email_source", "confidence", "status", "nodes_path", "errors", "timestamp",
]


def _format_email_source(best_email: dict) -> str:
    source = (best_email.get("source") or "").strip()
    note = (best_email.get("note") or "").strip()
    if source and note:
        return f"{source} - {note}"
    return note or source


def normalize_status(raw_status: str, email: str) -> str:
    # Perplexity removed → a no-website lead can exit as 'pending'. Collapse
    # anything that isn't a real hit to 'not_found'.
    if email:
        return "email_found"
    if raw_status == LeadStatus.FAILED.value:
        return "failed"
    return "not_found"


def result_to_sheet_row(raw_row: dict, result: dict) -> list:
    best = result.get("best_email") or {}
    if isinstance(best, dict):
        email = best.get("email", "")
        source_display = _format_email_source(best)
        confidence = best.get("confidence", "")
    else:
        email, source_display, confidence = "", "", ""

    nodes = result.get("nodes_executed") or []
    errors = result.get("errors") or []
    return [
        raw_row.get("Channel Name", ""),
        raw_row.get("_channel_url", ""),
        raw_row.get("_existing_email", ""),
        email,
        source_display,
        str(confidence),
        normalize_status(result.get("status", ""), email),
        " -> ".join(nodes),
        "; ".join(errors) if errors else "",
        datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
    ]


def _create_sheet_tab(sheet_service, spreadsheet_id, sheet_name):
    ok, names = sheet_service.list_sheets(spreadsheet_id)
    if ok and sheet_name in names:
        return
    sheet_service.service.spreadsheets().batchUpdate(
        spreadsheetId=spreadsheet_id,
        body={"requests": [{"addSheet": {"properties": {"title": sheet_name}}}]},
    ).execute()
    print(f"Created sheet tab '{sheet_name}'")


def _ensure_output_headers(sheet_service, spreadsheet_id, sheet_name):
    _create_sheet_tab(sheet_service, spreadsheet_id, sheet_name)
    ok, values = sheet_service.get_sheet_values(spreadsheet_id, f"{sheet_name}!A1:J1")
    if not ok or not values:
        sheet_service.append_rows(spreadsheet_id, sheet_name, [OUTPUT_HEADERS])
        print(f"Added headers to {sheet_name}")

In [ ]:
def run_pipeline(spreadsheet_id, limit=None, batch_size=BATCH_SIZE,
                 concurrency=CONCURRENCY, dry_run=True):
    """Run the Perplexity-free email finder over SOURCE_SHEET.

    Every processed channel (found or not) is written to OUTPUT_SHEET so the tab
    is a complete report. Resumable via CHECKPOINT_FILE (keyed by channel handle).

    Args:
        limit:   process at most this many channels (small test run).
        dry_run: run the agent but do NOT write to the sheet.
    """
    sheet_service = GoogleSheetService()
    cp = load_checkpoint(spreadsheet_id)
    already = set(cp["processed_keys"])

    print(f"Scanning '{SOURCE_SHEET}'...")
    all_rows = collect_rows(sheet_service, spreadsheet_id, already)
    if limit is not None:
        all_rows = all_rows[:limit]
        print(f"  (limited to first {len(all_rows)} for this run)")
    if not all_rows:
        print("Nothing to process.")
        return cp

    if not dry_run:
        _ensure_output_headers(sheet_service, spreadsheet_id, OUTPUT_SHEET)

    node_stats, source_stats = Counter(), Counter()
    total_batches = (len(all_rows) + batch_size - 1) // batch_size

    for bstart in range(0, len(all_rows), batch_size):
        batch = all_rows[bstart: bstart + batch_size]
        bnum = bstart // batch_size + 1
        print(f"\nBatch {bnum}/{total_batches} - {len(batch)} channels")

        results = run_batch_no_pplx(batch, source_type=SOURCE_TYPE,
                                    concurrency=concurrency, youtube_list=YOUTUBE_LIST)

        out_rows, out_keys = [], []
        for raw_row, result in zip(batch, results):
            email = (result.get("best_email") or {}).get("email", "")
            status = normalize_status(result.get("status", ""), email)
            for node in (result.get("nodes_executed") or []):
                node_stats[node] += 1
            best = result.get("best_email") or {}
            if isinstance(best, dict) and best.get("source"):
                source_stats[best["source"].split(" - ")[0]] += 1
            cp[f"{status}_count" if status in ("email_found", "failed") else "not_found_count"] = \
                cp.get(f"{status}_count" if status in ("email_found", "failed") else "not_found_count", 0) + 1
            out_rows.append(result_to_sheet_row(raw_row, result))
            out_keys.append(raw_row["_dedup_key"])

        if dry_run:
            print("  [dry_run] sample output rows:")
            for r in out_rows[:5]:
                print("   ", r[0], "|", r[3] or "(no email)", "|", r[6], "|", r[7])
            cp["processed_keys"].extend(out_keys)
            continue

        ok, msg = sheet_service.append_rows(spreadsheet_id, OUTPUT_SHEET, out_rows)
        print(f"  -> Wrote {len(out_rows)} rows to {OUTPUT_SHEET}: {msg}")
        if ok:
            cp["processed_keys"].extend(out_keys)
            save_checkpoint(cp)
        else:
            print("  WARNING sheet write failed - batch will be retried next run")

    print(f"\n{'='*50}\nPipeline complete\n{'='*50}")
    print(f"  Found:     {cp.get('email_found_count', 0)}")
    print(f"  Not found: {cp.get('not_found_count', 0)}")
    print(f"  Failed:    {cp.get('failed_count', 0)}")
    if node_stats:
        print("\n  Node execution counts:")
        for node, c in node_stats.most_common():
            print(f"    {node}: {c}")
    if source_stats:
        print("\n  Email sources:")
        for s, c in source_stats.most_common():
            print(f"    {s}: {c}")
    return cp

## Preview — confirm access, columns, and how many rows need an email

In [ ]:
sheet_service = GoogleSheetService()
spreadsheet_id = sheet_service.extract_spreadsheet_id(SPREADSHEET_URL)

ok, df = sheet_service.get_sheet_data(spreadsheet_id, SOURCE_SHEET)
assert ok, df
print(f"'{SOURCE_SHEET}': {len(df)} rows")
print(f"Columns: {list(df.columns)}")
need = sum(0 if is_real_email(e) else 1 for e in df["email"].tolist())
print(f"Rows that already have a real email: {len(df) - need}")
print(f"Rows needing an email:               {need}")

# Sanity-check the hygiene on a couple of rows.
for _, r in df.head(3).iterrows():
    raw = build_raw_row(r.to_dict())
    print("  →", raw["Channel Name"], "| url:", raw["_channel_url"],
          "| website:", raw.get("Website", "(dropped)"),
          "| email:", raw.get("Email", "(none)"))

## Single-channel smoke test (no sheet writes)

In [ ]:
sample = build_raw_row({
    "title": "Marques Brownlee",
    "channel_handle": "@mkbhd",
    "email": "x",
    "website_url": "https://i.ytimg.com/vi/abc/hqdefault.jpg",  # infra junk → dropped
    "country": "US",
    "niche": "tech",
})
print("raw_row:", json.dumps(sample, indent=2))

result = run_single_no_pplx(sample)
print("\nStatus:", result.get("status"))
print("Email: ", result.get("best_email", {}))
print("Path:  ", " -> ".join(result.get("nodes_executed") or []))
# Path should NOT contain 'perplexity_discovery' or 'fb_crawler'.

## Benchmark — 50 rows, real timing + ETA (no writes)

Run this **first** to get honest numbers before any scaled run. It reports the
per-lead timing distribution, how many leads actually hit the crawler, where
emails come from, and an extrapolated ETA for the full ~17.3k. Use those numbers
to decide the crawl strategy (crawl-everything vs only-rows-with-a-real-website).

In [ ]:
import statistics

BENCH_N = 50
sheet_service = GoogleSheetService()
spreadsheet_id = sheet_service.extract_spreadsheet_id(SPREADSHEET_URL)
bench_rows = collect_rows(sheet_service, spreadsheet_id, set())[:BENCH_N]
print(f"\nBenchmarking {len(bench_rows)} rows (concurrency={CONCURRENCY}, "
      f"per-lead cap={PER_LEAD_TIMEOUT_S}s)...\n")

LEAD_TIMINGS.clear()
t0 = time.monotonic()
bench_results = run_batch_no_pplx(bench_rows, concurrency=CONCURRENCY)
wall = time.monotonic() - t0

node_stats, source_stats = Counter(), Counter()
found = crawled = timed_out = 0
for res in bench_results:
    nodes = res.get("nodes_executed") or []
    for n in nodes:
        node_stats[n] += 1
    if "crawl_page" in nodes:
        crawled += 1
    if "__timeout__" in nodes:
        timed_out += 1
    if (res.get("best_email") or {}).get("email", ""):
        found += 1
        src = (res.get("best_email") or {}).get("source", "").split(" - ")[0]
        source_stats[src] += 1

ts = sorted(LEAD_TIMINGS)
def _pct(p): return ts[min(len(ts) - 1, int(len(ts) * p))] if ts else 0
print(f"\n{'='*56}\nBenchmark: {len(bench_results)} rows @ concurrency={CONCURRENCY}\n{'='*56}")
print(f"  Wall clock:        {wall:.0f}s  ({wall/len(bench_results):.1f}s/row effective)")
print(f"  Per-lead seconds:  min={ts[0]:.0f}  median={statistics.median(ts):.0f}  "
      f"p90={_pct(0.9):.0f}  max={ts[-1]:.0f}")
print(f"  Emails found:      {found}/{len(bench_results)}")
print(f"  Leads that crawled:{crawled}")
print(f"  Per-lead timeouts: {timed_out}")
print(f"  Node counts:       {dict(node_stats.most_common())}")
print(f"  Email sources:     {dict(source_stats.most_common())}")

NEED = 17300
eta_h = (wall / len(bench_results)) * NEED / 3600
print(f"\n  Extrapolated ETA for ~{NEED} rows @ concurrency={CONCURRENCY}: ~{eta_h:.1f} hours")
print("  Lever: raise CONCURRENCY (cuts ETA ~linearly until network/CPU-bound),")
print("  or only crawl the ~2.7k rows with a real website to skip Playwright for the rest.")

## Test run — first 10 channels

Starts in **dry-run** (no writes). Review the sample rows, then set
`dry_run=False` to actually write to the `Email_Finder` tab.

In [ ]:
run_pipeline(spreadsheet_id, limit=10, dry_run=True)

In [ ]:
# Same 10, now writing to the Email_Finder tab:
# run_pipeline(spreadsheet_id, limit=10, dry_run=False)

## Full / scaled run

Bump `limit` (or set `None` for everything that needs an email, ~17.3k rows).
The checkpoint makes it resumable — re-running only processes channels not yet
in `processed_keys`. Start with a few hundred to gauge hit-rate and cost before
committing to the whole list.

In [ ]:
# run_pipeline(spreadsheet_id, limit=500, dry_run=False)
# run_pipeline(spreadsheet_id, limit=None, dry_run=False)   # full run

# clear_checkpoint()   # uncomment to start over from scratch